<a href="https://colab.research.google.com/github/msadikozbek/Niyyah-Hadith-an-Levensthein-Difference-Map/blob/main/Niyyah_Hadith_and_Levensthein_Difference_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re
from enum import Enum
from IPython.display import HTML, display

# ============================================================
# 1. DATASET
# ============================================================

RIVAYETLER = {
    "Bad' al-Wahy, 1 (1/191)": {
        "kod": "B000001",
        "ravi": "al-Humaydi > Sufyan b. Uyayna > Yahya b. Sa'id",
        "metin": "إِنَّمَا الأَعْمَالُ بِالنِّيَّاتِ وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى دُنْيَا يُصِيبُهَا أَوْ إِلَى امْرَأَةٍ يَنْكِحُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ"
    },
    "Ayman wa-l-Nudhur, 23 (2/609)": {
        "kod": "B006689",
        "ravi": "Qutayba b. Sa'id > Abd al-Wahhab > Yahya b. Sa'id",
        "metin": "إِنَّمَا الأَعْمَالُ بِالنِّيَّةِ وَإِنَّمَا لاِمْرِئٍ مَا نَوَى فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ فَهِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ وَمَنْ كَانَتْ هِجْرَتُهُ إِلَى دُنْيَا يُصِيبُهَا أَوِ امْرَأَةٍ يَتَزَوَّجُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ"
    },
    "Itq, 6 (1/693)": {
        "kod": "B002529",
        "ravi": "Muhammad b. Kathir > Sufyan al-Thawri > Yahya b. Sa'id",
        "metin": "الأَعْمَالُ بِالنِّيَّةِ وَلاِمْرِئٍ مَا نَوَى فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ فَهِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ وَمَنْ كَانَتْ هِجْرَتُهُ لِدُنْيَا يُصِيبُهَا أَوِ امْرَأَةٍ يَتَزَوَّجُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ"
    },
    "Hiyal, 1 (2/661)": {
        "kod": "B006953",
        "ravi": "Abu al-Nu'man > Hammad b. Zayd > Yahya b. Sa'id",
        "metin": "يَا أَيُّهَا النَّاسُ إِنَّمَا الأَعْمَالُ بِالنِّيَّةِ وَإِنَّمَا لاِمْرِئٍ مَا نَوَى فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ فَهِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ وَمَنْ هَاجَرَ إِلَى دُنْيَا يُصِيبُهَا أَوِ امْرَأَةٍ يَتَزَوَّجُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ"
    },
    "Iman, 41 (1/206)": {
        "kod": "B000054",
        "ravi": "Abd Allah b. Maslama > Malik b. Anas > Yahya b. Sa'id",
        "metin": "الأَعْمَالُ بِالنِّيَّةِ وَلِكُلِّ امْرِئٍ مَا نَوَى فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ فَهِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ وَمَنْ كَانَتْ هِجْرَتُهُ لِدُنْيَا يُصِيبُهَا أَوِ امْرَأَةٍ يَتَزَوَّجُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ"
    },
    "Nikah, 5 (2/327)": {
        "kod": "B005070",
        "ravi": "Yahya b. Qaza'a > Malik b. Anas > Yahya b. Sa'id",
        "metin": "الْعَمَلُ بِالنِّيَّةِ وَإِنَّمَا لاِمْرِئٍ مَا نَوَى فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ فَهِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ صلى الله عليه وسلم وَمَنْ كَانَتْ هِجْرَتُهُ إِلَى دُنْيَا يُصِيبُهَا أَوِ امْرَأَةٍ يَنْكِحُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ"
    },
    "Manaqib al-Ansar, 45 (2/28)": {
        "kod": "B003898",
        "ravi": "Musaddad > Hammad b. Zayd > Yahya b. Sa'id",
        "metin": "الأَعْمَالُ بِالنِّيَّةِ فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى دُنْيَا يُصِيبُهَا أَوِ امْرَأَةٍ يَتَزَوَّجُهَا فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ وَمَنْ كَانَتْ هِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ فَهِجْرَتُهُ إِلَى اللَّهِ وَرَسُولِهِ"
    }
}

REFERANS = "Bad' al-Wahy, 1 (1/191)"

# ============================================================
# 2. NORMALIZATION & DISPLAY HELPERS
# ============================================================

class NormStep(Enum):
    HAMZA = "hamza"
    ALIF_MAQSURA = "alif_maqsura"
    TA_MARBUTA = "ta_marbuta"
    DIACRITICS = "diacritics"

def normalize(text, steps=None):
    if steps is None: steps = list(NormStep)
    r = text
    # Harekeleri temizleme
    if NormStep.DIACRITICS in steps:
        r = re.sub(r'[\u064B-\u065F\u0670]', '', r)
        r = r.replace('\u0640', '')
    # Hemzeleri standartlaştırma
    if NormStep.HAMZA in steps:
        r = r.replace('أ','ا').replace('إ','ا').replace('آ','ا')
        r = r.replace('ؤ','و').replace('ئ','ي')
    # Elif-i Maksure
    if NormStep.ALIF_MAQSURA in steps:
        r = r.replace('ى','ي')
    # Ta-i Marbuta
    if NormStep.TA_MARBUTA in steps:
        r = r.replace('ة','ه')
    return re.sub(r'\s+', ' ', r).strip()

def get_display_words(text):
    """Ekranda göstermek için orijinal metni kelimelere böler."""
    return re.sub(r'\s+', ' ', text).strip().split()

# ============================================================
# 3. LEVENSHTEIN LOGIC
# ============================================================

class OpType(Enum):
    MATCH = "match"
    INSERT = "insertion"
    DELETE = "deletion"
    REPLACE = "substitution"

class LevResult:
    def __init__(self, dist, ops, s1, s2):
        self.distance = dist; self.ops = ops; self.s1 = s1; self.s2 = s2
    @property
    def similarity(self):
        mx = max(len(self.s1), len(self.s2))
        return round((1 - self.distance/mx)*100, 1) if mx else 100.0
    @property
    def ins_count(self): return sum(1 for _,t in self.ops if t==OpType.INSERT)
    @property
    def del_count(self): return sum(1 for _,t in self.ops if t==OpType.DELETE)
    @property
    def sub_count(self): return sum(1 for _,t in self.ops if t==OpType.REPLACE)

def levenshtein(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1]==s2[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    ops = []; i, j = m, n
    while i>0 or j>0:
        if i>0 and j>0 and s1[i-1]==s2[j-1]:
            ops.append(((i-1,j-1),OpType.MATCH)); i-=1; j-=1
        elif i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+1:
            ops.append(((i-1,j-1),OpType.REPLACE)); i-=1; j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+1:
            ops.append(((i-1,-1),OpType.DELETE)); i-=1
        elif j>0 and dp[i][j]==dp[i][j-1]+1:
            ops.append(((-1,j-1),OpType.INSERT)); j-=1
        else: break
    ops.reverse()
    return LevResult(dp[m][n], ops, s1, s2)

def word_backtrace(w1, w2):
    m, n = len(w1), len(w2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if w1[i-1]==w2[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    raw = []; i, j = m, n
    while i>0 or j>0:
        if i>0 and j>0 and w1[i-1]==w2[j-1]:
            raw.append(("match",i-1,j-1)); i-=1; j-=1
        elif i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+1:
            raw.append(("sub",i-1,j-1)); i-=1; j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+1:
            raw.append(("del",i-1,-1)); i-=1
        elif j>0 and dp[i][j]==dp[i][j-1]+1:
            raw.append(("ins",-1,j-1)); j-=1
        else: break
    raw.reverse()
    return raw

# ============================================================
# 4. INLINE HTML DISPLAY
# ============================================================

def show_visual():
    ref_raw = RIVAYETLER[REFERANS]["metin"]
    ref_norm = normalize(ref_raw)

    html = ["""
<div style="font-family: 'Palatino Linotype', Palatino, serif; font-size: 11pt; padding: 20px; background: #ffffff; color: #222; line-height: 1.2; max-width: 1000px; margin: 0 auto; border: 1px solid #ddd; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
<h1 style="text-align: center; font-size: 15pt; color: #1a237e; margin-bottom: 4px;">Niyyah Hadith — Levenshtein Difference Map<br>
<small style="font-size: 11pt; font-weight: normal; color: #555;">Seven Transmissions in al-Bukhari's al-Jami' al-Sahih</small></h1>
<div style="text-align: center; margin: 8px 0 16px; font-size: 10pt;">
  <span style="background: #A5D6A7; padding: 2px 8px; border-radius: 3px; margin: 0 4px; border: 1px solid #ccc;">Insertion (ziyada)</span>
  <span style="background: #EF9A9A; padding: 2px 8px; border-radius: 3px; margin: 0 4px; border: 1px solid #ccc; text-decoration: line-through;">Deletion (nuqsan)</span>
  <span style="background: #FFF176; padding: 2px 8px; border-radius: 3px; margin: 0 4px; border: 1px solid #ccc;">Substitution (ibdal)</span>
</div>

<style>
  .ar .ins, .ar .del, .ar .sub { padding: 0; margin: 0; border-radius: 0; display: inline; border: none; }
</style>
"""]

    html.append(f"""
<div style="background: #f9f9f9; border: 1px solid #ddd; border-radius: 4px; padding: 10px 14px; margin-bottom: 12px;">
<div style="font-size: 11.5pt; font-weight: bold; color: #333; margin-bottom: 6px;">
    Reference Text: {REFERANS} <span style="font-weight: normal; color: #555; font-size: 10pt;">&mdash; Isnad: {RIVAYETLER[REFERANS]["ravi"]}</span>
</div>
<div class="ar" style="font-family: 'Sakkal Majalla', 'Traditional Arabic', serif; font-size: 15pt; direction: rtl; text-align: right; line-height: 1.5;">{ref_raw}</div>
<div style="font-size: 8.5pt; color: #999; margin-top: 6px;">Normalized ({len(ref_norm)} characters, {len(ref_norm.split())} words)</div>
</div>
""")

    for name, info in RIVAYETLER.items():
        if name == REFERANS: continue
        tgt_raw = info["metin"]

        w1_norm = ref_norm.split()
        w2_norm = normalize(tgt_raw).split()
        w1_disp = get_display_words(ref_raw)
        w2_disp = get_display_words(tgt_raw)

        raw_ops = word_backtrace(w1_norm, w2_norm)

        h_tgt_list = []
        wd_count = 0

        for op, i1, i2 in raw_ops:
            if op == "match":
                h_tgt_list.append(w2_disp[i2])
            elif op == "sub":
                wd_count += 1
                h_tgt_list.append(f'<span class="sub" style="background:#FFF176;">{w2_disp[i2]}</span>')
            elif op == "del":
                wd_count += 1
                h_tgt_list.append(f'<span class="del" style="background:#EF9A9A; text-decoration:line-through;">{w1_disp[i1]}</span>')
            elif op == "ins":
                wd_count += 1
                h_tgt_list.append(f'<span class="ins" style="background:#A5D6A7;">{w2_disp[i2]}</span>')

        h_tgt = " ".join(h_tgt_list)
        cr = levenshtein(ref_norm, normalize(tgt_raw))

        html.append(f"""
<div style="background: #fff; border: 1px solid #e0e0e0; border-radius: 4px; padding: 10px 14px; margin-bottom: 12px;">
<div style="font-size: 11pt; font-weight: bold; color: #283593; margin-bottom: 6px;">
    {name}
    <span style="font-weight: normal; color: #555; font-size: 9.5pt;">&mdash; Isnad: {info["ravi"]} &mdash;
    Distance: <strong style="color:#222;">{cr.distance}</strong> | Similarity: <strong style="color:#222;">{cr.similarity}%</strong> | Ins: <strong style="color:#222;">{cr.ins_count}</strong> Del: <strong style="color:#222;">{cr.del_count}</strong> Sub: <strong style="color:#222;">{cr.sub_count}</strong> | Word Diffs: <strong style="color:#222;">{wd_count}</strong></span>
</div>
<div class="ar" style="font-family: 'Sakkal Majalla', 'Traditional Arabic', serif; font-size: 15pt; direction: rtl; text-align: right; line-height: 1.5; padding: 6px; margin-bottom: 0px; background: #fafafa; border: 1px solid #eee; border-radius: 3px;">
    {h_tgt}
</div>
</div>
""")

    html.append("</div>")
    display(HTML("".join(html)))

if __name__=="__main__":
    show_visual()